# 03 — Smart Model: rubert-tiny2 fine-tune

Обучаем модель для тарифа **smart** (< 500ms, 3 кредита).

Артефакт на выходе: `models/smart_model/` (HuggingFace формат)

**Зависит от:** `data/train.csv`, `data/val.csv`, `data/test.csv` из ноутбука 01.

In [1]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [2]:
!pip install transformers datasets scikit-learn pandas torch -q

In [3]:
import pandas as pd
import numpy as np
import json, os
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch

In [4]:
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

CUDA available: True
GPU: Tesla T4
Device: cuda


In [5]:
os.makedirs('models/smart_model', exist_ok=True)

train = pd.read_csv('/kaggle/input/datasets/vegeradaria21/s-data/data/train.csv')
val   = pd.read_csv('/kaggle/input/datasets/vegeradaria21/s-data/data/val.csv')
test  = pd.read_csv('/kaggle/input/datasets/vegeradaria21/s-data/data/test.csv')

In [6]:
with open('/kaggle/input/datasets/vegeradaria21/s-data/data/id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}
with open('/kaggle/input/datasets/vegeradaria21/s-data/data/label2id.json') as f:
    label2id = json.load(f)

In [7]:
NUM_LABELS = 4
MODEL_NAME = 'cointegrated/rubert-tiny2'

print('Train:', len(train), '| Val:', len(val), '| Test:', len(test))
print('Классы:', train['label'].nunique())

Train: 11492 | Val: 2031 | Test: 2968
Классы: 4


## 1. Токенизатор и Dataset

In [8]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
train_dataset = IntentDataset(train, tokenizer)
val_dataset   = IntentDataset(val,   tokenizer)
test_dataset  = IntentDataset(test,  tokenizer)

print('Dataset shape:', train_dataset[0]['input_ids'].shape)

Dataset shape: torch.Size([128])


## 2. Модель

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
model = model.to(DEVICE)

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

In [11]:
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Параметров всего:   {total_params:,}')
print(f'Обучаемых:          {trainable:,}')

Параметров всего:   29,195,020
Обучаемых:          29,195,020


## 3. Обучение через HuggingFace Trainer

In [12]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro')),
    }

In [13]:
training_args = TrainingArguments(
    output_dir='models/smart_model_checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.990371,0.845406,0.889217,0.887695
2,0.593422,0.526464,0.916297,0.925010
3,0.461865,0.453085,0.926637,0.932140
4,0.391644,0.434671,0.929099,0.934400
5,0.362296,0.426899,0.931068,0.937180


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=900, training_loss=0.7705306641260783, metrics={'train_runtime': 78.6288, 'train_samples_per_second': 730.775, 'train_steps_per_second': 11.446, 'total_flos': 105958108139520.0, 'train_loss': 0.7705306641260783, 'epoch': 5.0})

## 4. Финальная оценка на тесте

In [14]:
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = test['label'].values

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [15]:
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='macro')

print(f'Test Accuracy: {acc:.4f}')
print(f'Test F1 macro: {f1:.4f}')
print()
print(classification_report(
    y_true, y_pred,
    target_names=[id2label[i] for i in range(NUM_LABELS)]
))

Test Accuracy: 0.9336
Test F1 macro: 0.9365

                    precision    recall  f1-score   support

account_management       0.93      0.95      0.94      1017
     entertainment       0.92      0.93      0.93       614
   general_inquiry       0.94      0.92      0.93      1117
   technical_issue       0.95      0.95      0.95       220

          accuracy                           0.93      2968
         macro avg       0.94      0.94      0.94      2968
      weighted avg       0.93      0.93      0.93      2968



## 5. Сохранение артефактов

In [16]:
model.save_pretrained('models/smart_model')
tokenizer.save_pretrained('models/smart_model')

metrics = {
    'accuracy': round(float(acc), 4),
    'f1_macro': round(float(f1), 4),
    'model': 'rubert-tiny2 fine-tuned',
    'base_model': MODEL_NAME,
    'n_train': len(train),
    'n_test': len(test),
    'classes': [id2label[i] for i in range(NUM_LABELS)],
    'max_input_length': 128,
    'ood_threshold': 0.55,
}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
with open('models/smart_model/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Сохранено в models/smart_model/')
print(f'F1 macro: {f1:.4f} | Accuracy: {acc:.4f}')

Сохранено в models/smart_model/
F1 macro: 0.9365 | Accuracy: 0.9336


## 6. OOD-детекция (Out-of-Distribution)

По архитектуре: если `max_proba < 0.55` → задача помечается флагом `low_confidence = True`
и уходит в HITL-очередь. Проверяем порог на тестовых данных.

In [18]:
import torch.nn.functional as F

OOD_THRESHOLD = 0.55

# Считаем max_proba для тестовой выборки
all_proba = F.softmax(torch.tensor(preds_output.predictions), dim=-1).numpy()
max_proba = all_proba.max(axis=1)

In [19]:
low_conf_mask = max_proba < OOD_THRESHOLD
print(f'Порог OOD: {OOD_THRESHOLD}')
print(f'Примеров с low_confidence: {low_conf_mask.sum()} / {len(max_proba)} ({low_conf_mask.mean()*100:.1f}%)')
print(f'Средняя max_proba: {max_proba.mean():.3f}')

# Accuracy только на high-confidence предиктах
high_conf = ~low_conf_mask
if high_conf.sum() > 0:
    acc_hc = accuracy_score(y_true[high_conf], y_pred[high_conf])
    print(f'Accuracy на high-confidence ({high_conf.sum()} примеров): {acc_hc:.4f}')

Порог OOD: 0.55
Примеров с low_confidence: 54 / 2968 (1.8%)
Средняя max_proba: 0.931
Accuracy на high-confidence (2914 примеров): 0.9427


## 7. Скорость инференса (CPU, имитация продакшна)

In [20]:
import time

model.eval()
cpu_model = model.cpu()

In [21]:
def predict_smart(text: str):
    """Полный пайплайн инференса — точная копия worker/ml/smart_predictor.py"""
    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding=True,
    )
    with torch.no_grad():
        out = cpu_model(**inputs)
        proba = F.softmax(out.logits, dim=-1)[0].numpy()
    idx = int(np.argmax(proba))
    confidence = float(proba[idx])
    low_conf = confidence < OOD_THRESHOLD
    return id2label[idx], confidence, low_conf

In [23]:
test_texts = [
    'Хочу вернуть товар',
    'Не работает приложение',
    'Списали деньги дважды',
    'Взломали мой аккаунт',
    'Требую руководителя',
    'Привет как дела',         
    'хочу пиццу',              
]

times = []
for text in test_texts:
    t0 = time.perf_counter()
    intent, conf, low_conf = predict_smart(text)
    elapsed = (time.perf_counter() - t0) * 1000
    times.append(elapsed)
    ood_flag = ' [LOW CONF → HITL]' if low_conf else ''
    status = 'DONE' if elapsed < 500 else '✗ МЕДЛЕННО'
    print(f'[{elapsed:5.0f}ms] {status}  {text!r:35s} → {intent} ({conf:.3f}){ood_flag}')

print(f'\nСреднее: {np.mean(times):.0f}ms | Макс: {np.max(times):.0f}ms')

[    9ms] DONE  'Хочу вернуть товар'                → account_management (0.723)
[    6ms] DONE  'Не работает приложение'            → general_inquiry (0.854)
[    7ms] DONE  'Списали деньги дважды'             → account_management (0.824)
[    6ms] DONE  'Взломали мой аккаунт'              → account_management (0.978)
[    5ms] DONE  'Требую руководителя'               → account_management (0.792)
[    5ms] DONE  'Привет как дела'                   → general_inquiry (0.929)
[    5ms] DONE  'хочу пиццу'                        → entertainment (0.953)

Среднее: 6ms | Макс: 9ms
